## 0. Install dependencies
Run this cell once, then restart the kernel.

In [1]:
# Run once, then restart kernel
!pip install bertopic sentence-transformers umap-learn hdbscan gensim pyarrow

  Using cached bertopic-0.17.4-py3-none-any.whl.metadata (24 kB)
  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
  Using cached umap_learn-0.5.12-py3-none-any.whl.metadata (24 kB)
  Using cached plotly-6.7.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached transformers-5.5.4-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.11.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-win_amd64.whl.metadata (7.4 kB)
  Using cached safetensors-0.7.0-cp38-abi3-win_amd64.whl.metadata (4.2 kB)
  Using cached fsspec-2026.3.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached pynndescent-0.6.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached smart_open-7.6.0-py3-none-any.whl.metadata (25 kB)
  Using cached tzdata-2026.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached narwhals-2.19.0-py3-non

## 1. Imports & setup

In [1]:
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

# For coherence scoring
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

import warnings
warnings.filterwarnings('ignore')

# Output directory
Path("results").mkdir(exist_ok=True)

print("All imports OK")

c:\Users\kiera\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports OK


## 2. Load data

In [ ]:
# edit the path as needed
df = pd.read_parquet("C:\\Users\\kiera\\OneDrive\\Desktop\\JHU\\Spring 2026\\NLP4CSS\\wildchat-demographics-analysis\\data\\wildchat_bertopic.parquet")

print(f"Rows: {len(df):,}")
print(f"States: {df['state'].nunique()}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nText length stats (chars):")
print(df['user_text'].str.len().describe().round(1))

# The documents list — this is what BERTopic takes as input
docs = df['user_text'].tolist()

Rows: 68,668
States: 36
Columns: ['conversation_hash', 'model', 'timestamp', 'hashed_ip', 'user_text', 'state']

Text length stats (chars):
count    68668.0
mean       545.3
std        751.2
min          1.0
25%         62.0
50%        158.0
75%        669.0
max      26765.0
Name: user_text, dtype: float64


## 3. Compute embeddings

In [4]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
EMBEDDINGS_CACHE = "results/embeddings_cache.npy"

if os.path.exists(EMBEDDINGS_CACHE):
    print("Loading cached embeddings...")
    embeddings = np.load(EMBEDDINGS_CACHE)
    print(f"Loaded embeddings: {embeddings.shape}")
else:
    print("Computing embeddings (this takes ~5-10 min on CPU, ~1-2 min on GPU)...")
    embedding_model = SentenceTransformer(EMBEDDING_MODEL)
    embeddings = embedding_model.encode(
        docs,
        show_progress_bar=True,
        batch_size=256,
        convert_to_numpy=True
    )
    np.save(EMBEDDINGS_CACHE, embeddings)
    print(f"Saved embeddings: {embeddings.shape}")

Computing embeddings (this takes ~5-10 min on CPU, ~1-2 min on GPU)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15633.96it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 269/269 [10:54<00:00,  2.43s/it]


Saved embeddings: (68668, 384)


## 4. Helper: coherence & diversity scoring

In [5]:
def compute_coherence(topic_model, docs, top_n=10):
    """
    Compute c_v coherence using gensim.
    Returns NaN if fewer than 2 valid topics.
    """
    topic_words = []
    for topic_id in topic_model.get_topics():
        if topic_id == -1:  # skip outlier topic
            continue
        words = [w for w, _ in topic_model.get_topic(topic_id)][:top_n]
        if words:
            topic_words.append(words)

    if len(topic_words) < 2:
        return float('nan')

    tokenized = [doc.lower().split() for doc in docs]
    dictionary = Dictionary(tokenized)

    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenized,
        dictionary=dictionary,
        coherence='c_v'
    )
    return cm.get_coherence()


def compute_diversity(topic_model, top_n=10):
    """
    Topic diversity: proportion of unique words across all topic top-N word lists.
    Higher = more diverse topics.
    """
    all_words = []
    for topic_id in topic_model.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in topic_model.get_topic(topic_id)][:top_n]
        all_words.extend(words)

    if not all_words:
        return float('nan')
    return len(set(all_words)) / len(all_words)


def outlier_rate(topics):
    """Fraction of documents assigned to the outlier topic (-1)."""
    topics = np.array(topics)
    return (topics == -1).sum() / len(topics)


print("Scoring functions defined.")

Scoring functions defined.


## 5. Parameter grid sweep

We sweep over:
- `min_topic_size`: minimum documents per topic (controls granularity)
- `n_neighbors` (UMAP): local neighborhood size (smaller = more local structure)
- `min_cluster_size` (HDBSCAN): minimum cluster size

Each run is scored on coherence, diversity, topic count, and outlier rate.
Target: **20–60 topics**, outlier rate < 20%, coherence > 0.4.

In [ ]:
# ── Parameter grid ────────────────────────────────────────────────────────────
# Keeping small for a first sweep, can add more values after seeing the range
GRID = [
    {"min_topic_size": 50,  "n_neighbors": 10, "min_cluster_size": 50},
    {"min_topic_size": 50,  "n_neighbors": 15, "min_cluster_size": 50},
    {"min_topic_size": 100, "n_neighbors": 10, "min_cluster_size": 100},
    {"min_topic_size": 100, "n_neighbors": 15, "min_cluster_size": 100},
    {"min_topic_size": 200, "n_neighbors": 15, "min_cluster_size": 200},
]

# Fixed settings across all runs
UMAP_N_COMPONENTS = 5   # standard for BERTopic
RANDOM_STATE = 42

# ── Run grid ──────────────────────────────────────────────────────────────────
experiment_log = []

for i, params in enumerate(GRID):
    run_id = f"run_{i+1:02d}"
    print(f"\n{'='*60}")
    print(f"{run_id}: {params}")
    print('='*60)

    umap_model = UMAP(
        n_neighbors=params["n_neighbors"],
        n_components=UMAP_N_COMPONENTS,
        min_dist=0.0,
        metric='cosine',
        random_state=RANDOM_STATE
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=params["min_cluster_size"],
        metric='euclidean',
        cluster_selection_method='eom',
        prediction_data=True
    )

    vectorizer_model = CountVectorizer(
        stop_words='english',
        min_df=5,
        max_df=0.95,
        ngram_range=(1, 2)
    )

    topic_model = BERTopic(
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        min_topic_size=params["min_topic_size"],
        verbose=True
    )

    topics, probs = topic_model.fit_transform(docs, embeddings=embeddings)

    n_topics = len(set(topics)) - (1 if -1 in topics else 0)
    out_rate = outlier_rate(topics)

    print(f"  Topics: {n_topics} | Outlier rate: {out_rate:.1%}")
    print("  Computing coherence (may take a minute)...")

    coherence = compute_coherence(topic_model, docs)
    diversity = compute_diversity(topic_model)

    print(f"  Coherence (c_v): {coherence:.4f} | Diversity: {diversity:.4f}")

    # Save model for candidate runs (topic count in useful range)
    if 15 <= n_topics <= 80:
        model_path = f"results/bertopic_{run_id}"
        topic_model.save(model_path)
        print(f"  Model saved → {model_path}")

    row = {
        "run_id": run_id,
        **params,
        "n_topics": n_topics,
        "outlier_rate": round(out_rate, 4),
        "coherence_cv": round(coherence, 4),
        "diversity": round(diversity, 4),
    }
    experiment_log.append(row)

log_df = pd.DataFrame(experiment_log)
log_df.to_csv("results/bertopic_experiment_log.csv", index=False)
print("\n\nExperiment log saved.")
print(log_df.to_string(index=False))


run_01: {'min_topic_size': 50, 'n_neighbors': 10, 'min_cluster_size': 50}


2026-04-19 15:31:34,374 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-19 15:32:31,614 - BERTopic - Dimensionality - Completed ✓
2026-04-19 15:32:31,616 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-19 15:32:39,083 - BERTopic - Cluster - Completed ✓
2026-04-19 15:32:39,098 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-19 15:32:45,963 - BERTopic - Representation - Completed ✓


  Topics: 184 | Outlier rate: 44.3%
  Computing coherence (may take a minute)...


2026-04-19 15:33:24,741 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm


  Coherence (c_v): 0.5667 | Diversity: 0.8043

run_02: {'min_topic_size': 50, 'n_neighbors': 15, 'min_cluster_size': 50}


...

  Model saved → results/bertopic_run_04

run_05: {'min_topic_size': 200, 'n_neighbors': 15, 'min_cluster_size': 200}


2026-04-19 15:38:45,932 - BERTopic - Dimensionality - Completed ✓
2026-04-19 15:38:45,935 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-19 15:38:51,545 - BERTopic - Cluster - Completed ✓
2026-04-19 15:38:51,554 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-19 15:38:58,246 - BERTopic - Representation - Completed ✓


  Topics: 37 | Outlier rate: 39.4%
  Computing coherence (may take a minute)...


2026-04-19 15:39:27,498 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


  Coherence (c_v): 0.5837 | Diversity: 0.8459
  Model saved → results/bertopic_run_05


Experiment log saved.
run_id  min_topic_size  n_neighbors  min_cluster_size  n_topics  outlier_rate  coherence_cv  diversity
run_01              50           10                50       184        0.4431        0.5667     0.8043
run_02              50           15                50       172        0.4555        0.5727     0.8116
run_03             100           10               100        78        0.4242        0.5667     0.8372
run_04             100           15               100        76        0.4493        0.5806     0.8408
run_05             200           15               200        37        0.3944        0.5837     0.8459


## 6. Select best run & inspect topics

Look at the experiment log above. Pick the run with:
- Topic count in a useful range (20–60 is ideal for this corpus size)
- Highest coherence
- Outlier rate < ~20%
- Good diversity (>0.7)

Update `BEST_RUN` below.

I am picking run_04.

In [ ]:
BEST_RUN = "run_04"  # updated after reviewing log

# Load best model
best_model = BERTopic.load(f"results/bertopic_{BEST_RUN}")

# Re-run transform to get topic assignments
topics, probs = best_model.transform(docs, embeddings=embeddings)
df['topic'] = topics

# Topic info table
topic_info = best_model.get_topic_info()
print(topic_info.to_string(index=False))

2026-04-19 15:45:11,185 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-04-19 15:45:11,318 - BERTopic - Dimensionality - Completed ✓
2026-04-19 15:45:11,319 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-04-19 15:45:15,417 - BERTopic - Cluster - Completed ✓


 Topic  Count                                                Name                                                                                                     Representation                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [8]:
# Print top words per topic
for topic_id in sorted(best_model.get_topics().keys()):
    if topic_id == -1:
        continue
    words = [w for w, _ in best_model.get_topic(topic_id)][:10]
    print(f"Topic {topic_id:3d}: {', '.join(words)}")

Topic   0: script, write script, vs, state, round, conference, write, pm, university, bowl
Topic   1: natsuki, monika, sayori, yuri, mc, just, club, literature, baby, school
Topic   2: translate, english, chinese, japanese, mean, spanish, does, mandarin, word, latin
Topic   3: political, empire, government, states, war, roman, did, peoples, president, united
Topic   4: detailed, ar, description, prompt, detailed description, coffee, ai, description scenes, scenes, lens
Topic   5: player, female, snow, just, strange, girl, spider, biome, blazette, base
Topic   6: probability, numbers, solve, angle, answer, triangle, equation, minus, bracket, number
Topic   7: fish, pond, fart, story, comedic, vividly, farts, water, buff, butt
Topic   8: management, business, customer, resume, experience, job, skills, project, work, team
Topic   9: email, bank, security, funds, presidio, anonymized, payment, presidio anonymized, cameron, address
Topic  10: story, write story, man, write, bully, extremely

## 7. Save topic-document matrix

In [9]:
df['topic'] = topics

# Save full topic assignments
doc_matrix = df[['conversation_hash', 'state', 'topic']].copy()
doc_matrix.to_parquet("results/topic_doc_matrix.parquet", index=False)
print(f"Saved topic_doc_matrix.parquet — {len(doc_matrix):,} rows")

Saved topic_doc_matrix.parquet — 68,668 rows


## 8. State-level topic proportions
This is the output needed for Spearman correlation analysis.

In [10]:
# Exclude outlier topic (-1) from proportions
df_valid = df[df['topic'] != -1].copy()

# Count topic assignments per state
state_topic_counts = (
    df_valid.groupby(['state', 'topic'])
    .size()
    .unstack(fill_value=0)
)

# Normalize to proportions (row-wise)
state_topic_props = state_topic_counts.div(state_topic_counts.sum(axis=1), axis=0)

# Rename columns to topic_0, topic_1, etc.
state_topic_props.columns = [f"topic_{c}" for c in state_topic_props.columns]
state_topic_props = state_topic_props.reset_index()

state_topic_props.to_parquet("results/state_topic_proportions.parquet", index=False)
print(f"Saved state_topic_proportions.parquet")
print(f"Shape: {state_topic_props.shape}  ({state_topic_props.shape[0]} states × {state_topic_props.shape[1]-1} topics)")
print(state_topic_props.head())

Saved state_topic_proportions.parquet
Shape: (36, 77)  (36 states × 76 topics)
        state   topic_0   topic_1   topic_2   topic_3   topic_4   topic_5  \
0     Alabama  0.000000  0.000000  0.000000  0.000000  0.032258  0.000000   
1     Arizona  0.015025  0.000000  0.040067  0.013356  0.075125  0.000000   
2    Arkansas  0.016807  0.000000  0.042017  0.008403  0.016807  0.067227   
3  California  0.005943  0.001358  0.064527  0.019019  0.162676  0.003736   
4    Colorado  0.008197  0.000000  0.028689  0.016393  0.008197  0.000000   

    topic_6   topic_7   topic_8  ...  topic_66  topic_67  topic_68  topic_69  \
0  0.000000  0.032258  0.032258  ...  0.000000  0.000000  0.000000  0.000000   
1  0.026711  0.003339  0.036728  ...  0.145242  0.001669  0.000000  0.011686   
2  0.033613  0.000000  0.025210  ...  0.000000  0.000000  0.008403  0.008403   
3  0.023434  0.001358  0.026660  ...  0.006283  0.002377  0.002377  0.005264   
4  0.004098  0.000000  0.073770  ...  0.000000  0.000000  

## 9. Representative prompts per topic
Top 10 prompts closest to each topic centroid (for topic labeling).

In [11]:
rep_rows = []

for topic_id in sorted(best_model.get_topics().keys()):
    if topic_id == -1:
        continue

    topic_docs = df[df['topic'] == topic_id]

    # Sample up to 10 representative prompts
    sample = topic_docs.sample(min(10, len(topic_docs)), random_state=42)

    top_words = ', '.join([w for w, _ in best_model.get_topic(topic_id)][:8])

    for _, row in sample.iterrows():
        rep_rows.append({
            'topic_id': topic_id,
            'top_words': top_words,
            'topic_label': '',  # Person D fills this in
            'user_text': row['user_text'],
            'state': row['state'],
        })

rep_df = pd.DataFrame(rep_rows)
rep_df.to_csv("results/representative_prompts.csv", index=False)
print(f"Saved representative_prompts.csv — {len(rep_df)} rows across {rep_df['topic_id'].nunique()} topics")
print("\nSample:")
print(rep_df[['topic_id', 'top_words', 'user_text']].head(5).to_string(index=False))

Saved representative_prompts.csv — 760 rows across 76 topics

Sample:
 topic_id                                                     top_words                                                                                                  user_text
        0 script, write script, vs, state, round, conference, write, pm                                                  write a script about what if big west schools were people
        0 script, write script, vs, state, round, conference, write, pm                                                              write a script about houston vs the big three
        0 script, write script, vs, state, round, conference, write, pm                                                      script about kansas state the 1990 national champions
        0 script, write script, vs, state, round, conference, write, pm                                         script about judy macleod adding her alma matter to the conference
        0 script, write script, vs,

## 10. Visualizations

In [12]:
# Topic word scores barchart
best_model.visualize_barchart(top_n_topics=20)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'marker': {'color': '#D55E00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.05321494290467687, 0.05961333181272941, 0.06207427923092895,
                    0.09758056398045227, 0.1744524598821191],
              'xaxis': 'x',
              'y': [round  , state  , vs  , write script  , script  ],
              'yaxis': 'y'},
             {'marker': {'color': '#0072B2'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.029073854158630105, 0.0575197528569545,
                    0.061219857824153734, 0.06597289944238945, 0.06900506675388053],
              'xaxis': 'x2',
              'y': [mc  , yuri  , sayori  , monika  , natsuki  ],
              'yaxis': 'y2'},
             {'marker': {'color': '#CC79A7'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.041032952155897244, 0.04837920404900048,
                    0.06497996268478863, 0.10916172646871683, 0.18442881350886609],
              'xaxis': 'x3',
              'y': [mean  , japanese  , chinese  , english  , translate  ],
              'yaxis': 'y3'},
             {'marker': {'color': '#E69F00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.020044745173671365, 0.02155199246037517,
                    0.025442432518130685, 0.028156708888292706,
                    0.028515726185405564],
              'xaxis': 'x4',
              'y': [war  , states  , government  , empire  , political  ],
              'yaxis': 'y4'},
             {'marker': {'color': '#56B4E9'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.01722655475360506, 0.01745348422214043, 0.0178820951932998,
                    0.019076028053501542, 0.019479774258980412],
              'xaxis': 'x5',
              'y': [detailed description  , prompt  , description  , ar  ,
                    detailed  ],
              'yaxis': 'y5'},
             {'marker': {'color': '#009E73'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.017510391450436136, 0.020199110966764425,
                    0.023228010673366255, 0.03252338634343525, 0.0817851835517258],
              'xaxis': 'x6',
              'y': [strange  , just  , snow  , female  , player  ],
              'yaxis': 'y6'},
             {'marker': {'color': '#F0E442'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.030905457005190654, 0.03129836020448818,
                    0.03131992257445442, 0.032479663204038736, 0.03322409730856865],
              'xaxis': 'x7',
              'y': [answer  , angle  , solve  , numbers  , probability  ],
              'yaxis': 'y7'},
             {'marker': {'color': '#D55E00'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.06803702408483316, 0.06930166660727213, 0.07533884331683967,
                    0.09418146179953003, 0.13255688314150518],
              'xaxis': 'x8',
              'y': [comedic  , story  , fart  , pond  , fish  ],
              'yaxis': 'y8'},
             {'marker': {'color': '#0072B2'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.01797500632378518, 0.019002213210170312,
                    0.019470465582136824, 0.019929991837423305,
                    0.021258167323513235],
              'xaxis': 'x9',
              'y': [experience  , resume  , customer  , business  , management  ],
              'yaxis': 'y9'},
             {'marker': {'color': '#CC79A7'},
              'orientation': 'h',
              'type': 'bar',
              'x': [0.027337186076798307, 0.027808392897147385,
                    0.027985759058448977, 0.03967350981901109, 0.05605386029284196],
              'xaxis': 'x10',
              'y': [presidio  , funds  , security  , bank  , email  ],
              'yaxis': 'y10'},
             {'marker': {'color': '#

In [13]:
# Topic similarity heatmap
best_model.visualize_heatmap()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'coloraxis': 'coloraxis',
              'hovertemplate': 'x: %{x}<br>y: %{y}<br>Similarity Score: %{z}<extra></extra>',
              'name': '0',
              'type': 'heatmap',
              'x': [0_script_write script_vs, 1_natsuki_monika_sayori,
                    2_translate_english_chinese, 3_political_empire_government,
                    4_detailed_ar_description, 5_player_female_snow,
                    6_probability_numbers_solve, 7_fish_pond_fart,
                    8_management_business_custo..., 9_email_bank_security,
                    10_story_write story_man, 11_tommy_girlfriend_ed,
                    12_mass_substance_chemical, 13_write 10_tweets_beach,
                    14_stock_rate_price, 15_music_genre_pop, 16_int_string_array,
                    17_god_bible_gods, 18_manipulation_physiology_...,
                    19_patients_treatment_cells, 20_models_ai_neural,
                    21_message_friend_write, 22_const_div_html,
                    23_email_thank_write email, 24_file_python_exe,
                    25_detailed_description_det..., 26_jewelry_necklace_ring,
                    27_lane_new jersey_jersey, 28_dick_batman_superman,
                    29_fight_women_melissa, 30_hi_buddy_nice meet,
                    31_gpt_chatgpt_chat, 32_hi_hello_whats, 33_food_cheese_recipe,
                    34_story_book_novel, 35_hello hello_hello_hello ...,
                    36_consciousness_brain_memory, 37_dan_chatgpt_content,
                    38_essay_evidence_rhetorical, 39_song_lyrics_rap,
                    40_poem_love_hearts, 41_house_property_unit,
                    42_reply_package_email, 43_villains_write story_han...,
                    44_tom_monkey_robot, 45_cat_pet_cats,
                    46_names_television_channel, 47_true_false_cpu,
                    48_motion_velocity_inches, 49_2023_jan_year,
                    50_china_dynasty_chinese, 51_detailed_prompt_descript...,
                    52_natsuki_bark_desert, 53_sonic_game_games,
                    54_table_select_sql, 55_user_caption_descriptive,
                    56_therapy_health_mental, 57_private_public_void,
                    58_tesla_electric_battery, 59_shorts_fusion_casual,
                    60_detailed_description_det..., 61_short reply_reply_genera...,
                    62_reading_comprehension_ag..., 63_scp_dr_researcher,
                    64_galaxy_image_markdown, 65_earth_time_planet,
                    66_cast_buzz_mr, 67_joke_funny_jokes, 68_know_dean_onlyfans,
                    69_park_visit_travel, 70_art_artists_artist,
                    71_megumin_dialogue story_d..., 72_prompt_ar_coffee,
                    73_alliance_elf_city, 74_tattoo_butterfly_peacock,
                    75_wizard_school_sayori],
              'xaxis': 'x',
              'y': [0_script_write script_vs, 1_natsuki_monika_sayori,
                    2_translate_english_chinese, 3_political_empire_government,
                    4_detailed_ar_description, 5_player_female_snow,
                    6_probability_numbers_solve, 7_fish_pond_fart,
                    8_management_business_custo..., 9_email_bank_security,
                    10_story_write story_man, 11_tommy_girlfriend_ed,
                    12_mass_substance_chemical, 13_write 10_tweets_beach,
                    14_stock_rate_price, 15_music_genre_pop, 16_int_string_array,
                    17_god_bible_gods, 18_manipulation_physiology_...,
                    19_patients_treatment_cells, 20_models_ai_neural,
                    21_message_friend_write, 22_const_div_html,
                    23_email_thank_write email, 24_file_python_exe,
                    25_detailed_description_det..., 26_jewelry_necklace_ring,
                    27_lane_new jersey_jersey, 28_dick_batman_superman,
                    29_fight_women_melissa, 30_hi_buddy_nice meet,
                    31_gpt_chatgpt_chat, 32_hi_hello_whats, 33_

In [14]:
# 2D topic map (requires UMAP reduce to 2D)
best_model.visualize_topics()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'customdata': array([[0, 'script | write script | vs | state | round', 3312],
                                   [1, 'natsuki | monika | sayori | yuri | mc', 2828],
                                   [2, 'translate | english | chinese | japanese | mean', 2484],
                                   ...,
                                   [73, 'alliance | elf | city | sayori | dialogue story', 111],
                                   [74, 'tattoo | butterfly | peacock | headings | sleeve', 102],
                                   [75, 'wizard | school | sayori | balance | city', 101]],
                                  shape=(76, 3), dtype=object),
              'hovertemplate': '<b>Topic %{customdata[0]}</b><br>%{customdata[1]}<br>Size: %{customdata[2]}',
              'legendgroup': '',
              'marker': {'color': '#B0BEC5',
                         'line': {'color': 'DarkSlateGrey', 'width': 2},
                         'size': {'bdata': ('8AwMC7QJzAYWBgEGsASOBPYDkANbA+' ... 'gAhwCFAH4AfAB3AHEAcABvAGYAZQA='),
                                  'dtype': 'i2'},
                         'sizemode': 'area',
                         'sizeref': 2.07,
                         'symbol': 'circle'},
              'mode': 'markers',
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'type': 'scatter',
              'x': {'bdata': ('9PU0QF7IVUGYIlRAxgCVwMw4fEF4kk' ... 'Fs5lJBQx/Nv7fsUEEKNuM/bpdUQQ=='),
                    'dtype': 'f4'},
              'xaxis': 'x',
              'y': {'bdata': ('btBMQT/AkEFyi0lBdF18QfxcQkAutY' ... 'BA64dB+yTKQJ/piEFysRdAT0eQQQ=='),
                    'dtype': 'f4'},
              'yaxis': 'y'}],
    'layout': {'annotations': [{'showarrow': False,
                                'text': 'D1',
                                'x': np.float32(-14.338699),
                                'y': np.float32(7.264561),
                                'yshift': 10},
                               {'showarrow': False,
                                'text': 'D2',
                                'x': np.float32(2.926443),
                                'xshift': 10,
                                'y': np.float32(28.454996)}],
               'height': 650,
               'hoverlabel': {'bgcolor': 'white', 'font': {'family': 'Rockwell', 'size': 16}},
               'legend': {'itemsizing': 'constant', 'tracegroupgap': 0},
               'margin': {'t': 60},
               'shapes': [{'line': {'color': '#CFD8DC', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(2.926443),
                           'x1': np.float32(2.926443),
                           'y0': np.float32(-13.925874),
                           'y1': np.float32(28.454996)},
                          {'line': {'color': '#9E9E9E', 'width': 2},
                           'type': 'line',
                           'x0': np.float32(-14.338699),
                           'x1': np.float32(20.191586),
                           'y0': np.float32(7.264561),
                           'y1': np.float32(7.264561)}],
               'sliders': [{'active': 0,
                            'pad': {'t': 50},
                            'steps': [{'args': [{'marker.color': [['red',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                                  '#B0BEC5',
                                                       

## 11. Final summary

In [16]:
best_row = log_df[log_df['run_id'] == BEST_RUN].iloc[0]

print("=" * 50)
print("FINAL BERTOPIC SUMMARY")
print("=" * 50)
print(f"Best run       : {BEST_RUN}")
print(f"Parameters     : min_topic_size={best_row['min_topic_size']}, n_neighbors={best_row['n_neighbors']}")
print(f"Topics found   : {best_row['n_topics']}")
print(f"Outlier rate   : {best_row['outlier_rate']:.1%}")
print(f"Coherence (cv) : {best_row['coherence_cv']:.4f}")
print(f"Diversity      : {best_row['diversity']:.4f}")
print("")
print("Output files:")
print("  results/bertopic_experiment_log.csv")
print(f"  results/bertopic_{BEST_RUN}/  (saved model)")
print("  results/topic_doc_matrix.parquet")
print("  results/state_topic_proportions.parquet")
print("  results/representative_prompts.csv")

FINAL BERTOPIC SUMMARY
Best run       : run_04
Parameters     : min_topic_size=100, n_neighbors=15
Topics found   : 76
Outlier rate   : 44.9%
Coherence (cv) : 0.5806
Diversity      : 0.8408

Output files:
  results/bertopic_experiment_log.csv
  results/bertopic_run_04/  (saved model)
  results/topic_doc_matrix.parquet
  results/state_topic_proportions.parquet
  results/representative_prompts.csv
